# Mapping Biology in Space
## DBiT-seq and the Basics of Spatial Multi-Omics Analysis

**V4SDB Research School 2026 — hands-on tutorial**

---

In this notebook you will load a real **spatial tri-omic** mouse-brain dataset — chromatin
accessibility (ATAC), gene expression (RNA), and protein (ADT) measured on the *same tissue
section, on the same physical pixels* — and check whether the biology lands where anatomy says
it should.

**Dataset.** Zhang, D. *et al.* "Spatial dynamics of brain development and neuroinflammation."
*Nature* **647**, 213–227 (2025). doi:[10.1038/s41586-025-09663-y](https://doi.org/10.1038/s41586-025-09663-y)

**Sample.** Mouse brain, postnatal day **P21**, replicate **S1**, spatial **ARP-seq**
(ATAC + RNA + Protein), 20 µm pixels on a 100 × 100 barcode grid.

**How to run this.** Top to bottom, one cell at a time (`Shift+Enter`). Nothing is installed on
your own machine and no data needs to be downloaded by hand — every cell fetches what it needs
from public archives.

> **You do not need to be a programmer to follow this.** Most cells you simply *run and read*.
> A handful are marked **🔧 Your turn** — there you change one value and re-run to see what
> happens. If a cell ever errors, it is almost always fine to just run it again; the notebook is
> designed to recover. Ask a demonstrator if you get stuck.

| Block | What you'll do | Approx. time |
|---|---|---|
| 0 | Set up the environment | 5 min |
| 0.5 | Python & Colab warm-up (skip if you code) | 10 min |
| 1 | Fetch the three modalities and build the data object | 15 min |
| 2 | Quality control, orientation, tissue-image overlay | 20 min |
| 2.5 | Cluster the pixels, then read the plots: bar, heatmap, dot plot, GO enrichment | 20 min |
| 3 | Plot RNA, ATAC and protein on the same grid | 25 min |
| 4 | Validate against known anatomy | 20 min |
| 5 | Capstone: compare P0 vs P21 and see the brain develop | 15 min |

### Where the data actually comes from

Worth knowing, because it is typical of how a modern multi-omic paper releases data — the
modalities live in **different archives**, and none of them is a single tidy download:

| Modality | Archive | What you get |
|---|---|---|
| RNA | GEO `GSE308526`, sample `GSM9247588` | gene × pixel count matrix (CSV, ~12 MB) |
| ATAC | GEO `GSE308599`, sample `GSM9249004` | fragment coordinates (`fragments.tsv.gz`, **879 MB**) |
| Protein (ADT) | NeMO Archive, grant `rf1_fan`, collection [`col-0cggtum`](https://assets.nemoarchive.org/col-0cggtum) | `05_P21S1_ADT_matrix.csv.tar` — protein × pixel matrix (~0.5 MB) |
| Pixel coordinates | GEO, with the RNA sample | `tissue_positions_list.csv` |
| CODEX imaging + code | Zenodo [10.5281/zenodo.17121652](https://doi.org/10.5281/zenodo.17121652) | microscopy images, processing scripts |

A wrinkle worth naming: unlike GEO, the **NeMO Archive has no per-sample accession** like a `GSM`
number. It organises data by *grant* (`rf1_fan`, the Fan lab) and *collection* (`col-0cggtum`),
and you locate a sample by its **file path** within that collection — here
`.../DBiT_protein-seq/mouse/processed/counts/05_P21S1_ADT_matrix.csv.tar`. The full URL is in
Block 1. That is simply how this archive is indexed; it is not missing information.

The ATAC file is far too large to download during a workshop. We use a trick instead —
see Block 3 — that reads **only the few kilobytes we actually need** straight out of the remote
file, without ever downloading the whole thing.

---
## Block 0 — Environment setup

One cell. It makes sure two packages are present in this notebook's runtime (not on your own
computer): `pysam`, which we need in Block 3 to read remote genomic files, and `scikit-learn`,
which we use to cluster pixels in Block 2.5. On Colab both installs are quick — `scikit-learn` is
usually already there. Everything else (`pandas`, `numpy`, `matplotlib`, `scipy`, `requests`) is
present by default.

*(The `%%capture` on the first line just hides a long, noisy install log. That is all it does.)*

In [ ]:
%%capture
!pip install -q pysam scikit-learn

In [ ]:
import io, json, os, re, tarfile, time
from collections import Counter

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 9,
                     "axes.titlesize": 9, "figure.facecolor": "white"})

print("Environment ready.")

---
## Block 0.5 — A two-minute Python & Colab warm-up

**Already comfortable with Python and notebooks? Skip straight to Block 1.**

If not, read this once and you will understand every cell that follows.

- A **notebook** is a column of *cells*. A cell is either **text** (like this one) or **code**
  (grey background). You run a code cell by clicking it and pressing **`Shift`+`Enter`**; its
  output appears just below it.
- Run the cells **in order, top to bottom.** Later cells depend on variables created earlier.
- **Errors are normal and not scary.** The important line is the **last** one — it names what
  went wrong. Usually the fix is "I skipped a cell above"; just run the cells in order.
- The one data structure we use everywhere is a **DataFrame** — a table with named rows and
  columns, from the `pandas` library. All three modalities are DataFrames.

Run the next cell to meet a tiny DataFrame before we touch the real, big one.

In [ ]:
# A toy "genes x pixels" table: 3 genes measured in 4 pixels. Numbers are made up.
toy = pd.DataFrame(
    [[0, 3, 0, 1],
     [5, 4, 6, 5],
     [0, 0, 2, 0]],
    index=["Mbp", "Snap25", "Gfap"],          # row labels  = genes
    columns=["pixel_1", "pixel_2", "pixel_3", "pixel_4"])  # column labels = pixels

print("shape (rows, columns):", toy.shape)   # -> (3 genes, 4 pixels)
print()
print(toy)                                    # look at the whole little table
print()
print("Counts for Mbp across the 4 pixels:")
print(toy.loc["Mbp"])                         # .loc[label] pulls out one row by name

That is genuinely most of what you need:

- `table.shape` → `(number of rows, number of columns)`.
- `table.loc["Mbp"]` → the row for one gene, as a list of numbers (one per pixel).
- `table.head()` → just the first few rows, handy when a table is huge.

The real RNA table you load in a moment is the same shape of object — just with ~20,000 genes
instead of 3, and ~9,000 pixels instead of 4.

#### 🔧 Your turn

In the cell below, change the gene name in the last line from `"Snap25"` to `"Gfap"` and re-run
(`Shift`+`Enter`). You should see the `Gfap` row instead. This is exactly the move you will make
on the real data later — pick a gene, look at it.

In [ ]:
gene = "Snap25"          # <-- change this to "Gfap" (or "Mbp") and re-run
print(f"{gene} across the 4 toy pixels:")
print(toy.loc[gene])

---
## Block 1 — Fetch the three modalities

Each modality is a **matrix of counts**: rows are features (genes, or proteins), columns are
**pixels**. A pixel is one square of the barcode grid — here 20 µm across, which at P21 is
roughly one to a few cells. Every pixel is identified by a DNA barcode, and the *same barcode
means the same physical square in all three modalities*. That shared barcode is what makes this
"tri-omic" rather than three separate experiments.

The next cell defines the download URLs and a small helper, `fetch()`, that downloads a file the
first time and then reuses the local copy. You do not need to read the helper line by line — just
run it and watch the files arrive.

In [ ]:
GEO_RNA   = ("https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM9247nnn/GSM9247588/suppl/"
             "GSM9247588_05_P21S1_RNA_matrix.csv.gz")
GEO_POS   = ("https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM9247nnn/GSM9247588/suppl/"
             "GSM9247588_05_P21S1_tissue_positions_list.csv.gz")
NEMO_ADT  = ("https://data.nemoarchive.org/biccn/grant/rf1_fan/fan/multimodal/bulk/"
             "DBiT_protein-seq/mouse/processed/counts/05_P21S1_ADT_matrix.csv.tar")
ATAC_FRAG = ("https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM9249nnn/GSM9249004/suppl/"
             "GSM9249004_05_P21S1_atac_fragments.tsv.gz")
ATAC_TBI  = ATAC_FRAG + ".tbi.gz"

os.makedirs("data", exist_ok=True)

def fetch(url, path):
    """Download `url` to `path`, skipping the download if the file is already there."""
    if os.path.exists(path):
        print(f"  cached  {os.path.basename(path)}")
        return path
    t0 = time.time()
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(path, "wb") as fh:
            for chunk in r.iter_content(1 << 20):
                fh.write(chunk)
    print(f"  fetched {os.path.basename(path)}  "
          f"({os.path.getsize(path)/1e6:.1f} MB, {time.time()-t0:.0f}s)")
    return path

print("Downloading RNA, pixel coordinates and protein ...")
rna_path = fetch(GEO_RNA,  "data/P21S1_RNA.csv.gz")
pos_path = fetch(GEO_POS,  "data/P21S1_positions.csv.gz")
adt_path = fetch(NEMO_ADT, "data/P21S1_ADT.tar")
print("Done.")

### The pixel coordinate file

`tissue_positions_list.csv` is the map from barcode to position. Its columns are the same ones
10x Genomics uses for Visium, which is why DBiT-seq data can be read by Visium-shaped tools:

`barcode, in_tissue, array_row, array_col, pixel_row, pixel_col`

`in_tissue` is the flag produced when someone looked at the microscope image and decided which
grid squares actually sit on the tissue. Squares off the tissue are dropped — they contain
nothing but background.

In [ ]:
positions = pd.read_csv(
    pos_path, header=None,
    names=["barcode", "in_tissue", "array_row", "array_col", "pixel_row", "pixel_col"])

print(f"Barcode grid      : {positions.array_row.max()+1} x {positions.array_col.max()+1} "
      f"= {len(positions):,} squares")
print(f"On tissue         : {int((positions.in_tissue == 1).sum()):,}")
print(f"Off tissue (drop) : {int((positions.in_tissue == 0).sum()):,}")

positions = positions[positions.in_tissue == 1].set_index("barcode")
positions.head()

In [ ]:
# RNA: genes x pixels. Downcast to int32: counts never exceed a few thousand, and
# halving the matrix's memory matters on Colab's ~12 GB runtime.
rna = pd.read_csv(rna_path, index_col=0).astype(np.int32)

# Protein: the NeMO download is a .tar wrapping a .csv.gz
with tarfile.open(adt_path) as tar:
    member = next(m for m in tar.getmembers() if m.name.endswith(".csv.gz"))
    adt = pd.read_csv(io.BytesIO(tar.extractfile(member).read()),
                      index_col=0, compression="gzip")

print(f"RNA     : {rna.shape[0]:,} genes    x {rna.shape[1]:,} pixels")
print(f"Protein : {adt.shape[0]:,} proteins x {adt.shape[1]:,} pixels")

### Line up the three tables on a common set of pixels

The RNA and protein matrices do not necessarily carry the same pixels in the same order — the
protein matrix here still contains all 10,000 grid squares, including the off-tissue ones. So we
take the **intersection**: barcodes that are on the tissue *and* present in both matrices. This
"align on shared barcodes" step is the practical heart of multi-omic analysis, and getting it
wrong silently scrambles everything downstream.

In [ ]:
barcodes = [b for b in rna.columns if b in positions.index and b in adt.columns]

rna = rna[barcodes]
adt = adt[barcodes]
xy  = positions.loc[barcodes, ["array_col", "array_row"]].to_numpy(float)

# 'unmapped' is a technical row (reads matching no antibody), not a real protein
adt = adt.drop(index="unmapped", errors="ignore")

print(f"Shared pixels across all modalities: {len(barcodes):,}")
print(f"RNA  {rna.shape}   Protein  {adt.shape}   Coordinates  {xy.shape}")

#### 🔧 Your turn — look at your own data

You now have a real `rna` table. Before moving on, poke at it. Run the cell below as-is, then try
changing `"Mbp"` to another gene — `"Snap25"` (neurons), `"Gfap"` (astrocytes) or `"Plp1"`
(myelin) are all present. Notice most numbers are **0**: at this depth, any single gene is silent
in most pixels. That sparsity is the theme of the whole session.

In [ ]:
gene = "Mbp"                                   # <-- try "Snap25", "Gfap", "Plp1"
row = rna.loc[gene]
print(f"{gene}: detected in {int((row > 0).sum()):,} of {rna.shape[1]:,} pixels")
print(f"       total counts across the section: {int(row.sum()):,}")
row.head()

---
## Block 2 — Quality control and orientation

Two questions, always, before any biology:

1. **Did we get enough reads per pixel?** Spatial methods spread a sequencing run over
   thousands of pixels, so each pixel gets a small fraction of the depth a single cell would get
   in scRNA-seq. Low depth shows up as speckled, uninterpretable maps.
2. **Is the tissue the right way round?** The barcode grid has no inherent anatomical
   orientation. If it is flipped or rotated, every anatomical statement you make afterwards is
   wrong — and the plots will still look perfectly plausible.

> #### 🔮 Predict first
> The next cell draws two histograms (counts per pixel, genes per pixel) and a map of sequencing
> depth in space. Before you run it: do you expect **every** pixel to have the same depth? Or
> should denser tissue give more counts? Jot a guess, then run and check the third panel.

In [ ]:
counts_per_pixel = rna.sum(axis=0)
genes_per_pixel  = (rna > 0).sum(axis=0)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))

axes[0].hist(counts_per_pixel, bins=60, color="#4C72B0")
axes[0].set_xlabel("RNA counts per pixel"); axes[0].set_ylabel("pixels")
axes[0].axvline(counts_per_pixel.median(), color="crimson", ls="--",
                label=f"median {counts_per_pixel.median():,.0f}")
axes[0].legend()

axes[1].hist(genes_per_pixel, bins=60, color="#55A868")
axes[1].set_xlabel("genes detected per pixel"); axes[1].set_ylabel("pixels")
axes[1].axvline(genes_per_pixel.median(), color="crimson", ls="--",
                label=f"median {genes_per_pixel.median():,.0f}")
axes[1].legend()

sc = axes[2].scatter(xy[:, 0], -xy[:, 1], c=np.log10(counts_per_pixel + 1),
                     s=5, marker="s", cmap="viridis")
axes[2].set_aspect("equal"); axes[2].axis("off")
axes[2].set_title("log10 RNA counts, in space")
plt.colorbar(sc, ax=axes[2], shrink=0.8)

plt.tight_layout(); plt.show()

print(f"Median counts/pixel : {counts_per_pixel.median():,.0f}")
print(f"Median genes/pixel  : {genes_per_pixel.median():,.0f}")
print(f"Total RNA counts    : {counts_per_pixel.sum():,.0f}")

**Read the third panel carefully.** Depth is not uniform across the section, and it is not
supposed to be — denser tissue yields more material. What you are looking for is whether depth
varies *smoothly* with anatomy (fine) or shows stripes, corners or blocks (a technical problem
with the microfluidic channels).

Note the median of roughly 1,300 counts per pixel. That is perfectly usable, but it is one to
two orders of magnitude below a typical scRNA-seq cell — which is exactly why individual sparse
genes will look noisy later, and why we will need to be careful about it.

In [ ]:
def smooth_values(values, k=1):
    """Average each pixel with its neighbours within k grid squares.

    Smoothing trades spatial resolution for signal. It is useful for sparse measurements
    and dishonest if you forget you did it — so always report the value of k.
    """
    v = np.asarray(values, dtype=float)
    if not k:
        return v
    lookup = {(int(x), int(y)): v[i] for i, (x, y) in enumerate(xy)}
    return np.array([
        np.mean([lookup[(int(x)+dx, int(y)+dy)]
                 for dx in range(-k, k+1)
                 for dy in range(-k, k+1)
                 if (int(x)+dx, int(y)+dy) in lookup])
        for x, y in xy])


def spatial_panel(ax, values, title, cmap="viridis", smooth=0, diverging=False):
    """Plot one value per pixel on the barcode grid."""
    v = smooth_values(values, smooth)

    lo, hi = np.quantile(v, 0.02), np.quantile(v, 0.98)
    if diverging:
        bound = max(abs(lo), abs(hi)); lo, hi = -bound, bound
    if hi <= lo:
        lo, hi = float(v.min()), float(max(v.max(), v.min() + 1e-9))

    ax.scatter(xy[:, 0], -xy[:, 1], c=v, s=5.5, marker="s", cmap=cmap, vmin=lo, vmax=hi)
    ax.set_title(title); ax.set_aspect("equal"); ax.axis("off")
    return ax

def cp10k(counts):
    """Counts-per-10k + log1p normalisation, computed in float32.

    float32 halves the memory of the normalised matrix. On Colab's ~12 GB runtime
    that is the difference between running and crashing, and the precision is far
    beyond anything these counts justify anyway.
    """
    scale = (1e4 / counts.sum(axis=0)).astype(np.float32)
    return np.log1p(counts.astype(np.float32).mul(scale, axis=1))


# Orientation check: a gene whose anatomy you already know.
# Mbp marks myelin, so it should trace the corpus callosum - a bright arc, not a blob.
rna_norm = cp10k(rna)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
spatial_panel(axes[0], rna_norm.loc["Mbp"], "Mbp (RNA) — raw pixels")
spatial_panel(axes[1], rna_norm.loc["Mbp"], "Mbp (RNA) — smoothed", smooth=1)
plt.tight_layout(); plt.show()

That arc is the **corpus callosum**, the white-matter tract carrying axons between the two
hemispheres. Seeing it means the grid is oriented sensibly and the section is a coronal
hemisphere with cortex on the outside — so anatomical claims we make from here are on solid
ground.

Compare the two panels. Same data. The right-hand one is easier to read because each pixel has
been averaged with its eight neighbours; the cost is that fine structure has been blurred away.
Smoothing is a presentation choice, not an analysis result — always say when you have used it.

#### 🔧 Your turn — plot a gene with the *opposite* pattern

`Mbp` lights up the white matter. **Neurons** sit in the cortex above it, so a neuronal gene
should paint the opposite region. Run the cell below (it uses `Snap25`, a pan-neuronal marker),
then try `Rasgrf2` or `Cux2` — cortical-layer genes. You are looking for signal in the *cortex*,
i.e. the outer band, complementary to the `Mbp` arc.

<details><summary>💡 Hint / what you should see</summary>

Neuronal markers fill the broad cortical territory and are dim in the corpus-callosum arc — the
mirror image of `Mbp`. If you see the two patterns as complements, you have correctly oriented
the tissue *and* confirmed the biology, from two independent genes.
</details>

In [ ]:
gene = "Snap25"          # <-- try "Rasgrf2", "Cux2", "Satb2"
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
spatial_panel(axes[0], rna_norm.loc["Mbp"], "Mbp — myelin (reference)", smooth=1)
spatial_panel(axes[1], rna_norm.loc[gene], f"{gene} — smoothed", smooth=1)
plt.tight_layout(); plt.show()

### The tissue image underneath the pixels

Everything so far was drawn on the abstract barcode grid — but the section was **photographed
under the microscope before barcoding**, and GEO ships that image alongside the counts
(Visium convention: a `tissue_hires_image` plus a `scalefactors` file that maps pixel
coordinates into the image). Overlaying the two is worth doing once, for three reasons:

1. it shows where the `in_tissue` mask came from — someone drew it on exactly this image;
2. it is the ultimate orientation check — molecular signal should trace visible anatomy;
3. it reminds you that every dot in every plot is a real, physical square of this tissue.

We keep the remaining plots on the plain grid (faster, less visual noise), but the image is
always one cell away.

In [ ]:
import gzip

GEO_IMG = ("https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM9247nnn/GSM9247588/suppl/"
           "GSM9247588_05_P21S1_tissue_hires_image.png.gz")
GEO_SF  = ("https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM9247nnn/GSM9247588/suppl/"
           "GSM9247588_05_P21S1_scalefactors_json.json.gz")

# uint8 keeps the image at ~160 MB instead of ~650 MB as float32
img = (plt.imread(io.BytesIO(gzip.open(fetch(GEO_IMG, "data/P21S1_hires.png.gz"),
                                       "rb").read())) * 255).astype(np.uint8)
sf  = json.load(gzip.open(fetch(GEO_SF, "data/P21S1_scalefactors.json.gz"), "rt"))
scale = sf["tissue_hires_scalef"]
print(f"image {img.shape[0]} x {img.shape[1]} px, scale factor {scale}")

# pixel_row / pixel_col are image coordinates; the scale factor maps them onto this image
px = positions.loc[barcodes, ["pixel_col", "pixel_row"]].to_numpy(float) * scale

fig, axes = plt.subplots(1, 2, figsize=(9.6, 4.8))
axes[0].imshow(img)
axes[0].set_title("the tissue section, as imaged"); axes[0].axis("off")
axes[1].imshow(img)
axes[1].scatter(px[:, 0], px[:, 1], c=rna_norm.loc["Mbp"], s=2.5,
                cmap="viridis", alpha=0.75)
axes[1].set_title("Mbp (RNA) overlaid on the tissue"); axes[1].axis("off")
plt.tight_layout(); plt.show()

The bright arc sits exactly on the pale band visible in the photograph — molecular data and
histology agreeing about where the corpus callosum is. When these two disagree, it almost always
means a coordinate bug (flipped axis, wrong scale factor), and this overlay is how you catch it.

---
## Block 2.5 — Reading the plots you will meet everywhere

Spatial maps are only one of the chart types in an omics paper. The rest answer, in different
visual languages, *"which genes are high in which **groups** of pixels?"* — so first we need
groups. This is exactly what a real pipeline does: **cluster** the pixels by their whole
expression profile into candidate cell types / regions, then describe those clusters.

We do the standard thing (the same steps Scanpy or Seurat run under the hood):

1. keep the 1,000 most variable genes,
2. compress them to 20 principal components (PCA),
3. group pixels with **k-means** into 7 clusters.

No anatomy goes in — the clusters are found from expression alone. `scikit-learn` is already on
Colab, so this is a few seconds.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# 1-2. most-variable genes -> PCA (pixels x components)
hvg = rna_norm.var(axis=1).nlargest(1000).index
pcs = PCA(n_components=20, random_state=0).fit_transform(rna_norm.loc[hvg].T.to_numpy())

# 3. k-means into 7 clusters
K = 7
cluster = KMeans(n_clusters=K, n_init=10, random_state=0).fit_predict(pcs)

# Clusters come out in arbitrary order. Relabel them by mean cortical depth (grid row) so the
# plots below read surface -> deep. This uses position ONLY for ordering, not for clustering.
depth_of = {c: xy[cluster == c, 1].mean() for c in range(K)}
order = sorted(range(K), key=lambda c: depth_of[c])
relabel = {old: new for new, old in enumerate(order)}
cluster = np.array([relabel[c] for c in cluster])
cluster_names = [f"c{c}" for c in range(K)]

print("cluster sizes (surface -> deep):",
      [int((cluster == c).sum()) for c in range(K)])

Plot the clusters in space. Because we ordered them by depth, they should stack as bands from the
cortical surface inward — with one cluster picking out the corpus-callosum arc. That the *unbiased*
clustering recovers anatomy is the real result; the plots after this just describe it.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.4, 4.6))
for c in range(K):
    m = cluster == c
    axes[0].scatter(xy[m, 0], -xy[m, 1], s=6, marker="s", color=plt.cm.tab10(c),
                    label=cluster_names[c])
axes[0].set_aspect("equal"); axes[0].axis("off")
axes[0].set_title("k-means clusters, in space")
axes[0].legend(markerscale=1.6, fontsize=8, loc="center left", bbox_to_anchor=(1.0, 0.5))

# which cluster is white matter? the one with the highest mean Mbp
mbp = rna_norm.loc["Mbp"].to_numpy()
wm_cluster = int(np.argmax([mbp[cluster == c].mean() for c in range(K)]))
spatial_panel(axes[1], (cluster == wm_cluster).astype(float),
              f"cluster {wm_cluster}: highest Mbp = white matter", cmap="magma")
plt.tight_layout(); plt.show()
print(f"White-matter cluster is c{wm_cluster}.")

Now the three chart types every omics paper leans on — each **built from these clusters**, so you
read them on something real rather than a textbook cartoon.

In [ ]:
FIGLIT_MARKERS = ["Cux2", "Rasgrf2", "Rorb", "Bcl11b", "Foxp2", "Tle4", "Mbp", "Plp1"]
FIGLIT_MARKERS = [g for g in FIGLIT_MARKERS if g in rna_norm.index]

# For each marker x cluster: mean normalised expression, and fraction of pixels expressing it.
mean_expr = np.zeros((len(FIGLIT_MARKERS), K))
frac_expr = np.zeros((len(FIGLIT_MARKERS), K))
for i, g in enumerate(FIGLIT_MARKERS):
    v   = rna_norm.loc[g].to_numpy(float)
    raw = rna.loc[g].to_numpy(float)
    for c in range(K):
        m = cluster == c
        mean_expr[i, c] = v[m].mean()
        frac_expr[i, c] = (raw[m] > 0).mean()

# Scale each gene to its own 0-1 range across clusters. Mbp is ~10x more abundant than any
# layer marker, so without this the few loud genes dominate the colour scale and everything
# else looks flat. Scaling per gene shows each marker's PATTERN. (Scanpy calls this
# standard_scale='var'; it is the default mental model for reading a marker heatmap.)
span = mean_expr.max(1, keepdims=True) - mean_expr.min(1, keepdims=True)
scaled_expr = (mean_expr - mean_expr.min(1, keepdims=True)) / np.where(span == 0, 1, span)
print("Built a per-cluster summary for", len(FIGLIT_MARKERS), "marker genes.")

### 1. Bar plot — the simplest possible chart

A **bar plot** encodes one number per category as bar length. Here: total signal per marker
across the whole section. The practical lesson — some textbook markers are simply *rare* at this
depth, and a bar plot makes that obvious at a glance.

In [ ]:
totals = rna.reindex(FIGLIT_MARKERS).sum(axis=1).sort_values()
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.barh(totals.index, totals.values, color="#4C72B0")
ax.set_xlabel("total RNA counts across the section")
ax.set_title("Bar plot — abundance of each marker gene")
plt.tight_layout(); plt.show()

### 2. Heatmap — a whole table as a grid of colours

A **heatmap** shows a matrix (markers × clusters) with **colour = value**. We colour by the
*per-gene scaled* expression (each row stretched to its own 0–1 range) so you can compare each
marker's **pattern** — otherwise `Mbp`, ten times louder than the layer markers, would wash the
rest of the plot flat. Read it by scanning each row for its brightest cluster: `Mbp`/`Plp1` peak
sharply in the white-matter cluster, and the cortical markers pick out different clusters along
the surface→deep ordering (more subtly — cortical layers are genuinely harder to separate at this
depth than white matter).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(scaled_expr, aspect="auto", cmap="magma", vmin=0, vmax=1)
ax.set_xticks(range(K)); ax.set_xticklabels(cluster_names)
ax.set_yticks(range(len(FIGLIT_MARKERS))); ax.set_yticklabels(FIGLIT_MARKERS)
ax.set_xlabel("cluster  (ordered surface → deep)")
ax.set_title("Heatmap — per-gene scaled expression, marker × cluster")
plt.colorbar(im, ax=ax, shrink=0.8, label="scaled expr (per gene, 0–1)")
plt.tight_layout(); plt.show()

### 3. Dot plot — a heatmap that shows *two* numbers at once

A **dot plot** is the workhorse of single-cell papers. Each dot encodes **two** things:

- **colour** = average expression (per-gene scaled, as in the heatmap), and
- **size** = the *fraction* of pixels in that cluster where the gene is detected at all.

That second channel matters: a gene can be bright but small (a few high pixels) or big (broadly
on). Read colour and size together.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4))
for i, g in enumerate(FIGLIT_MARKERS):
    for c in range(K):
        ax.scatter(c, i, s=20 + frac_expr[i, c] * 300,
                   c=[scaled_expr[i, c]], cmap="magma", vmin=0, vmax=1,
                   edgecolors="0.6", linewidths=0.4)
ax.set_xticks(range(K)); ax.set_xticklabels(cluster_names)
ax.set_yticks(range(len(FIGLIT_MARKERS))); ax.set_yticklabels(FIGLIT_MARKERS)
ax.set_xlabel("cluster  (ordered surface → deep)")
ax.set_title("Dot plot — colour = mean expr, size = fraction of pixels expressing")
ax.margins(0.08)
plt.tight_layout(); plt.show()

### 4. GO enrichment — from a cluster to its biology

The plots above used *known* markers. But clustering gives you clusters with **no labels** — so
the real workflow is the reverse: take a cluster, find the genes that are high in it, and ask
*what biological processes those genes belong to*. That is **gene-ontology (GO) enrichment**, and
its standard picture is another dot plot.

We do it live for the white-matter cluster: rank genes by (mean in cluster − mean elsewhere),
take the top 100, and send them to the **g:Profiler** web service. If the term list comes back
dominated by myelination, the pipeline has rediscovered the biology from scratch.

In [ ]:
# Top genes for the white-matter cluster: simple difference-of-means ranking.
X = rna_norm.to_numpy()
in_c = cluster == wm_cluster
score = X[:, in_c].mean(1) - X[:, ~in_c].mean(1)
wm_markers = pd.Series(score, index=rna_norm.index).nlargest(100).index.tolist()
print("Top white-matter genes:", wm_markers[:12], "...")

# Ask g:Profiler for GO biological-process enrichment. Wrapped in try/except so the notebook
# still runs if the service is unreachable (same spirit as the ATAC guard).
GO_OK = False
try:
    r = requests.post("https://biit.cs.ut.ee/gprofiler/api/gost/profile/",
                      json={"organism": "mmusculus", "query": wm_markers,
                            "sources": ["GO:BP"], "user_threshold": 0.05,
                            "no_evidences": True}, timeout=60)
    res = pd.DataFrame(r.json()["result"])
    res["gene_ratio"] = res["intersection_size"] / res["query_size"]
    res = res.nsmallest(10, "p_value")
    GO_OK = len(res) > 0
    print(f"g:Profiler returned {len(r.json()['result'])} enriched GO terms.")
except Exception as exc:
    print("g:Profiler unavailable, using a small offline example:", exc)
    res = pd.DataFrame({
        "name": ["myelination", "ensheathment of neurons", "axon ensheathment",
                 "glial cell differentiation", "oligodendrocyte development"],
        "p_value": [1e-18, 3e-17, 8e-9, 2e-11, 5e-10],
        "intersection_size": [14, 13, 9, 11, 8],
        "gene_ratio": [0.28, 0.26, 0.18, 0.22, 0.16]})

In [ ]:
res = res.sort_values("gene_ratio")
fig, ax = plt.subplots(figsize=(7.6, 3.8))
sc = ax.scatter(res["gene_ratio"], range(len(res)),
                s=res["intersection_size"] * 22, c=-np.log10(res["p_value"]),
                cmap="viridis", edgecolors="0.4", linewidths=0.5)
ax.set_yticks(range(len(res))); ax.set_yticklabels([n[:42] for n in res["name"]])
ax.set_xlabel("gene ratio  (query genes in the term / query size)")
ax.set_title("GO enrichment dot plot — white-matter cluster")
plt.colorbar(sc, ax=ax, shrink=0.85, label="-log10 p-value")
for s, lab in [(6 * 22, "6"), (14 * 22, "14 genes")]:
    ax.scatter([], [], s=s, c="0.6", edgecolors="0.4", label=lab)
ax.legend(scatterpoints=1, frameon=False, labelspacing=1.3, loc="lower right", fontsize=9)
plt.tight_layout(); plt.show()

Read this dot plot exactly like the expression one, but the rows are now **biological processes**:
x-position = gene ratio, dot size = how many of your genes hit the term, colour = significance.
The top terms are all myelination / ensheathment — the unlabelled cluster is oligodendrocytes,
recovered end to end from raw counts. That round trip — cluster → marker genes → GO terms → a
named cell type — is the backbone of almost every single-cell and spatial paper.

**Chart types, one fact.** Bar, heatmap, expression dot plot and GO dot plot all answer "what is
where?" in different languages. You will also meet **Sankey diagrams** (flows, e.g. cluster →
cell type → region); the reading rule never changes — find the axes, the colour scale, and, for
dot/bubble plots, what the *size* means.

---
## Block 3 — Three modalities, one grid

This is the part that makes tri-omics concrete. We will take **one gene, `Mbp`**, and ask three
different molecular questions about the *same physical squares of tissue*:

| Modality | The question it answers |
|---|---|
| **ATAC** | Is the `Mbp` locus *open* — is the cell allowed to read this gene? |
| **RNA** | Is `Mbp` actually being *transcribed*? |
| **Protein** | Has MBP protein *accumulated* here? |

RNA and protein are already loaded. ATAC needs one more step, because of that 879 MB file.

### Reading 879 MB without downloading 879 MB

The ATAC data is not a matrix. It is a list of **fragments**: every row is one piece of DNA the
enzyme cut out, recorded as chromosome, start, end, and which pixel barcode it came from.

```
chr18   82489225   82490028   CGCATACACCAGTTCA-1   1
```

To ask "how open is `Mbp` in each pixel?" we only need the fragments landing in the `Mbp` locus
— a few thousand rows out of hundreds of millions. The file is **bgzip-compressed and
tabix-indexed**, and the GEO server supports HTTP range requests, so we can download the small
index and then pull just the byte ranges covering our region. Each query takes about a second.

This is worth remembering well beyond this workshop: indexed genomic formats (BAM, VCF,
tabix-indexed TSV) can nearly always be queried remotely instead of downloaded.

*(If you are running this on the workshop server, the full fragments file is already on disk
next to the notebook and the same code just reads it locally — even faster. The cell below
detects which situation you are in automatically.)*

In [ ]:
import gzip, pysam

# The index is small (~0.7 MB). GEO stores it gzipped, so we unpack it locally.
fetch(ATAC_TBI, "data/P21S1_fragments.tbi.gz")
if not os.path.exists("data/P21S1_fragments.tbi"):
    with gzip.open("data/P21S1_fragments.tbi.gz", "rb") as fi, \
         open("data/P21S1_fragments.tbi", "wb") as fo:
        fo.write(fi.read())

# On the workshop server the full fragments file is pre-staged in data/, so queries
# are local and instant. Anywhere else (Colab, home) we query GEO remotely instead.
frag_source = "data/P21S1_fragments.tsv.gz"
if not os.path.exists(frag_source):
    frag_source = ATAC_FRAG

ATAC_OK = False
try:
    fragments = pysam.TabixFile(frag_source, index="data/P21S1_fragments.tbi")
    t0 = time.time()
    probe = sum(1 for _ in fragments.fetch("chr18", 82_475_000, 82_590_000))
    where = "local file" if frag_source != ATAC_FRAG else "remote GEO file, no bulk download"
    print(f"ATAC access working ({where}) — {probe:,} fragments at the Mbp locus "
          f"in {time.time()-t0:.1f}s.")
    ATAC_OK = True
except Exception as exc:
    print("Could not open the remote fragments file:", exc)
    print("RNA and protein below will still work; ATAC panels will be skipped.")

In [ ]:
def fragment_counts(chrom, start, end):
    """Fragments per pixel overlapping a genomic interval."""
    tally = Counter()
    for row in fragments.fetch(chrom, max(0, start), end):
        fields = row.split("\t")
        tally[fields[3].split("-")[0]] += 1          # strip the Cell-Ranger '-1' suffix
    return pd.Series(tally).reindex(barcodes).fillna(0.0)


def gene_locus(symbol, genome="mm10"):
    """Look up a gene's mm10 coordinates from the UCSC API."""
    hits = requests.get("https://api.genome.ucsc.edu/search",
                        params={"search": symbol, "genome": genome}, timeout=30).json()
    spans = []
    for block in hits.get("positionMatches", []):
        for match in block.get("matches", []):
            m = re.match(r"(chr[\w]+):(\d+)-(\d+)", match.get("position", ""))
            if m:
                spans.append((m.group(1), int(m.group(2)), int(m.group(3))))
    if not spans:
        return None

    chrom = Counter(s[0] for s in spans).most_common(1)[0][0]
    spans = [s for s in spans if s[0] == chrom]
    track = requests.get("https://api.genome.ucsc.edu/getData/track",
                         params={"genome": genome, "track": "ncbiRefSeqCurated",
                                 "chrom": chrom,
                                 "start": min(s[1] for s in spans),
                                 "end":   max(s[2] for s in spans)}, timeout=30).json()
    records = [r for r in track.get("ncbiRefSeqCurated", []) if r.get("name2") == symbol]
    if not records:
        return None

    start, end = min(r["txStart"] for r in records), max(r["txEnd"] for r in records)
    if end - start > 1_000_000:          # guard against an ambiguous name match
        return None
    return chrom, start, end


def gene_accessibility(symbol, flank=2_000):
    """Fragments per pixel over a gene body plus flanking sequence ('gene activity')."""
    locus = gene_locus(symbol)
    if locus is None:
        raise ValueError(f"could not resolve {symbol} in mm10")
    chrom, start, end = locus
    return fragment_counts(chrom, start - flank, end + flank)


if ATAC_OK:
    mbp_atac = gene_accessibility("Mbp")
    print(f"Mbp accessibility: {mbp_atac.sum():,.0f} fragments across "
          f"{int((mbp_atac > 0).sum()):,} pixels")

### Normalising each modality on its own terms

Three measurement types, three conventions — using the wrong one is a common and quiet mistake:

- **RNA** — counts per 10,000, then `log1p`. Standard for count data spanning orders of magnitude.
- **Protein** — **centred log-ratio (CLR)** across proteins within each pixel. ADT counts carry
  a large, pixel-specific background from unbound antibody; CLR compares each protein to the
  pixel's own average rather than to an absolute scale.
- **ATAC** — divide by each pixel's total accessibility. A pixel with more fragments overall will
  have more fragments everywhere, so raw counts mostly measure sequencing depth. We estimate that
  depth from `chr19` — the smallest mouse autosome, a fair proxy for genome-wide coverage and one
  more cheap tabix query.

In [ ]:
def clr(frame):
    """Centred log-ratio across features within each pixel."""
    x = frame.to_numpy(float) + 1.0
    return pd.DataFrame(np.log(x / np.exp(np.log(x).mean(axis=0))),
                        index=frame.index, columns=frame.columns)

# rna_norm was already computed in Block 2 with cp10k() — no need to recompute it.
adt_norm = clr(adt)

if ATAC_OK:
    atac_depth = fragment_counts("chr19", 0, 61_500_000)
    print(f"Median ATAC fragments per pixel on chr19: {atac_depth.median():,.0f}")

    def atac_norm(counts):
        scaled = counts / atac_depth.replace(0, np.nan) * atac_depth.median()
        return np.log1p(scaled).fillna(0.0)

> #### 🔮 Predict first
> The next cell puts `Mbp` side by side as RNA, ATAC and protein. Two of the three should show a
> clean corpus-callosum arc. **Which one do you expect to look noisiest, and why?** (Hint: look
> back at the median counts per pixel from Block 2, then remember a single ATAC locus gets only a
> handful of fragments.)

In [ ]:
n_panels = 3 if ATAC_OK else 2
fig, axes = plt.subplots(1, n_panels, figsize=(4.1 * n_panels, 4.2))

spatial_panel(axes[0], rna_norm.loc["Mbp"], "RNA — Mbp transcript", smooth=1)
spatial_panel(axes[-1], adt_norm.loc["MBP"], "Protein — MBP (ADT)",
              cmap="cividis", smooth=1)
if ATAC_OK:
    spatial_panel(axes[1], atac_norm(mbp_atac), "ATAC — Mbp locus accessibility",
                  cmap="magma", smooth=1)

fig.suptitle("Same tissue section, same pixels, three molecular layers", y=1.02)
plt.tight_layout(); plt.show()

**What you should see, and what you should not.**

RNA gives a crisp corpus callosum. Protein gives the same structure, broader and blurrier —
protein accumulates and diffuses along the axons it wraps, so it is not confined to the cell
bodies transcribing the gene. That difference is real biology, not a processing artefact, and it
is precisely what a single-modality experiment cannot show you.

The ATAC panel is the disappointing one, and it is worth being honest about why. A single gene
locus receives only a few thousand fragments spread over ~9,000 pixels, so per-pixel counts are
mostly zero or one — the sparsity problem from Block 2, in its sharpest form. The signal is
there, but it is buried in shot noise.

The fix is the subject of the next cell.

#### 🔧 Your turn — a different cell type

You just looked at `Mbp` (myelin). Change the gene in the cell below to see a different cell type
across RNA and protein. Good choices that have a matching antibody in the protein panel:
`Cux2` (upper-layer neurons) or `Satb2` (cortical projection neurons). The cell falls back to
RNA-only if a gene has no protein counterpart, so nothing breaks if you pick something else.

<details><summary>💡 Hint</summary>

Protein names are UPPERCASE in `adt_norm` (e.g. gene `Cux2` → protein `CUX2_CUX1`). The cell
handles that mapping for the suggested genes automatically.
</details>

In [ ]:
gene = "Cux2"            # <-- try "Satb2", "Mog", or your own choice
protein_of = {"Cux2": "CUX2_CUX1", "Satb2": "SATB2", "Mbp": "MBP", "Mog": "MOG"}
prot = protein_of.get(gene)

has_prot = prot in adt_norm.index if prot else False
n = 2 if has_prot else 1
fig, axes = plt.subplots(1, n, figsize=(4.1 * n, 4.2), squeeze=False)
spatial_panel(axes[0, 0], rna_norm.loc[gene], f"RNA — {gene}", smooth=1)
if has_prot:
    spatial_panel(axes[0, 1], adt_norm.loc[prot], f"Protein — {prot}",
                  cmap="cividis", smooth=1)
plt.tight_layout(); plt.show()

### Making sparse ATAC readable: aggregate a gene programme

Rather than asking about one gene, ask about a **set of genes that turn on together**. Summing
across a myelin programme multiplies the available fragments; comparing it against a neuronal
programme cancels the per-pixel depth term entirely, because both are measured in the same pixel.

That ratio — myelin accessibility versus neuronal accessibility — is what we plot. This is a
general and heavily used strategy: **when a single feature is too sparse, aggregate a biologically
coherent set of features.**

In [ ]:
MYELIN_PROGRAMME = ["Mbp", "Mog", "Plp1", "Cldn11", "Sox10", "Ugt8a", "Trf"]
NEURON_PROGRAMME = ["Snap25", "Syt1", "Nefl", "Camk2a", "Neurod6", "Slc17a7", "Satb2"]

def programme_accessibility(symbols):
    total = pd.Series(0.0, index=barcodes)
    for symbol in symbols:
        try:
            total = total + gene_accessibility(symbol)
        except Exception:
            print(f"  (skipped {symbol})")
    return total

if ATAC_OK:
    print("Querying myelin programme ...")
    myelin_atac = programme_accessibility(MYELIN_PROGRAMME)
    print("Querying neuronal programme ...")
    neuron_atac = programme_accessibility(NEURON_PROGRAMME)

    # log ratio: depth cancels, because both are counted in the same pixels
    atac_contrast = np.log2((myelin_atac + 1) / (neuron_atac + 1))

    rna_contrast = np.log2(
        (rna.reindex(MYELIN_PROGRAMME).dropna().sum() + 1) /
        (rna.reindex(NEURON_PROGRAMME).dropna().sum() + 1))

    print(f"\nMyelin programme  : {myelin_atac.sum():,.0f} fragments")
    print(f"Neuronal programme: {neuron_atac.sum():,.0f} fragments")

In [ ]:
if ATAC_OK:
    fig, axes = plt.subplots(1, 3, figsize=(12.5, 4.2))
    spatial_panel(axes[0], rna_norm.loc["Mbp"], "RNA — Mbp (reference)", smooth=1)
    spatial_panel(axes[1], rna_contrast, "RNA — myelin vs neuronal programme",
                  cmap="RdBu_r", smooth=1, diverging=True)
    spatial_panel(axes[2], atac_contrast, "ATAC — myelin vs neuronal programme",
                  cmap="RdBu_r", smooth=2, diverging=True)
    plt.tight_layout(); plt.show()

The corpus callosum now emerges from the **chromatin** data alone — the same structure, recovered
from a completely independent molecular layer. Nothing about the ATAC analysis used the RNA data,
so this is genuine cross-modal agreement rather than a circular result.

Note that we needed heavier smoothing (`smooth=2`) for ATAC than for RNA. That is the honest cost
of the sparser modality, and it is the kind of detail that belongs in a methods section.

#### 🔧 Your turn — build your own gene programme

The programme trick is not specific to myelin. Below is an **astrocyte** programme; astrocytes
are spread more evenly through the brain than myelin, so expect a subtler, more diffuse pattern
rather than a sharp arc. Run it as-is, then try editing the gene list — add `Slc1a3`, or build a
programme of your own. (This queries the remote ATAC file gene by gene, so it takes a few
seconds; it is skipped automatically if ATAC access failed.)

<details><summary>💡 Hint — a microglia programme to try instead</summary>

`["Cx3cr1", "C1qa", "C1qb", "Csf1r", "Ctss"]` — microglia are sparse and scattered, an even
harder case that shows why aggregation matters.
</details>

In [ ]:
MY_PROGRAMME = ["Gfap", "Aqp4", "Slc1a3"]      # <-- edit this list

if ATAC_OK:
    print("Querying your programme ...")
    my_atac = programme_accessibility(MY_PROGRAMME)
    my_contrast = np.log2((my_atac + 1) / (neuron_atac + 1))
    my_rna = np.log2(
        (rna.reindex(MY_PROGRAMME).dropna().sum() + 1) /
        (rna.reindex(NEURON_PROGRAMME).dropna().sum() + 1))
    fig, axes = plt.subplots(1, 2, figsize=(8.4, 4.2))
    spatial_panel(axes[0], my_rna, "RNA — your programme vs neuronal",
                  cmap="RdBu_r", smooth=1, diverging=True)
    spatial_panel(axes[1], my_contrast, "ATAC — your programme vs neuronal",
                  cmap="RdBu_r", smooth=2, diverging=True)
    plt.tight_layout(); plt.show()
else:
    print("ATAC access is off, so this exercise is skipped. Read on.")

---
## Block 4 — Does the biology land where anatomy says it should?

The mouse cortex is layered, and different genes mark different layers. At P21 those layers are
mature and should be plainly visible. This is the standard sanity check: **if known markers do
not land in known places, stop and fix the data — do not proceed to clustering.**

| Marker | Expected location |
|---|---|
| `Cux2`, `Rasgrf2` | upper cortical layers II–III |
| `Rorb` | layer IV |
| `Bcl11b` (CTIP2) | layer V |
| `Foxp2`, `Tle4` | layer VI |
| `Mbp`, `Plp1` | corpus callosum (white matter) |

A note on marker choice: the textbook deep-layer markers `Fezf2` and `Tbr1` are in this dataset,
but with only ~100 and ~150 total counts across all 8,988 pixels they are far too sparse to plot
usefully. `Foxp2` and `Tle4` mark the same deep layers with ten times the signal. Picking markers
that are both *specific* and *abundant enough at your sequencing depth* is a real skill, and the
first choice is not always the right one.

In [ ]:
LAYER_MARKERS = ["Cux2", "Rasgrf2", "Rorb", "Bcl11b", "Foxp2", "Tle4", "Mbp", "Plp1"]
available = [g for g in LAYER_MARKERS if g in rna_norm.index]

fig, axes = plt.subplots(2, 4, figsize=(13, 7))
for ax, gene in zip(axes.ravel(), available):
    spatial_panel(ax, rna_norm.loc[gene], f"{gene} (RNA)", smooth=1)
for ax in axes.ravel()[len(available):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

Read these as a set, not one at a time. `Cux2` and `Rasgrf2` hug the outer edge of the cortex,
`Rorb` sits just inside them, `Bcl11b` deeper still, `Foxp2` and `Tle4` deeper again, and
`Mbp`/`Plp1` mark the white matter beneath them all. Ordered outside-in, they stack into
concentric bands that follow the curve of the cortical surface.

**That ordering is the result.** Any single panel proves very little — a blob of signal somewhere
in the tissue is easy to produce by accident. Several markers landing in the correct relative
order is hard to produce by accident, and that is what makes this a real check rather than a
reassuring picture.

#### 🔧 Your turn — try a marker that *fails*

Swap one of the working markers for a textbook one that is **too sparse**: change `gene` below to
`"Fezf2"` or `"Tbr1"`. You should get a nearly empty panel. This is the point from the table
above, made concrete: a biologically correct marker can still be unusable if it is too rare at
your sequencing depth. Then try a good one (`"Rorb"`) to see the difference.

In [ ]:
gene = "Fezf2"           # <-- try "Tbr1" (sparse) then "Rorb" (abundant)
total = int(rna.loc[gene].sum()) if gene in rna.index else 0
fig, ax = plt.subplots(figsize=(4.2, 4.2))
if gene in rna_norm.index:
    spatial_panel(ax, rna_norm.loc[gene], f"{gene} — {total:,} total counts", smooth=1)
else:
    ax.set_title(f"{gene} not in dataset"); ax.axis("off")
plt.tight_layout(); plt.show()

### The same anatomy, seen in protein

The ~150-antibody panel includes direct counterparts to several of these markers. Comparing them
tests both the biology *and* the two independent measurement chemistries against each other.

In [ ]:
PAIRS = [("Mbp", "MBP"), ("Mog", "MOG"), ("Cux2", "CUX2_CUX1"),
         ("Tbr1", "TBR1"), ("Bcl11b", "CTIP2"), ("Satb2", "SATB2")]
pairs = [(g, p) for g, p in PAIRS if g in rna_norm.index and p in adt_norm.index]

fig, axes = plt.subplots(2, len(pairs), figsize=(2.3 * len(pairs), 5.2))
for i, (gene, protein) in enumerate(pairs):
    spatial_panel(axes[0, i], rna_norm.loc[gene],    f"{gene} — RNA", smooth=1)
    spatial_panel(axes[1, i], adt_norm.loc[protein], f"{protein} — protein",
                  cmap="cividis", smooth=1)
plt.tight_layout(); plt.show()

In [ ]:
# Quantify the agreement rather than eyeballing it.
from scipy.stats import spearmanr

print(f"{'gene':>8}  {'protein':>12}  {'rho (per pixel)':>16}  {'rho (smoothed)':>15}")
print("-" * 60)
for gene, protein in pairs:
    raw = spearmanr(rna_norm.loc[gene], adt_norm.loc[protein]).statistic
    sm  = spearmanr(smooth_values(rna_norm.loc[gene], 1),
                    smooth_values(adt_norm.loc[protein], 1)).statistic
    print(f"{gene:>8}  {protein:>12}  {raw:>16.3f}  {sm:>15.3f}")

if ATAC_OK:
    raw = spearmanr(atac_contrast, rna_contrast).statistic
    sm  = spearmanr(smooth_values(atac_contrast, 2),
                    smooth_values(rna_contrast, 1)).statistic
    print(f"\n{'ATAC vs RNA':>8}  {'(programme)':>12}  {raw:>16.3f}  {sm:>15.3f}")

**Two things to take from that table.**

First, the per-pixel numbers are *low* — roughly 0.03 to 0.35. Smoothing over each pixel's
immediate neighbours raises them substantially. Nothing about the biology changed between those
two columns; the per-pixel figure is dominated by counting noise, and averaging a handful of
neighbouring pixels averages much of that noise away. This is the sparsity problem from Block 2,
now measured rather than merely asserted.

Second, even smoothed, these are moderate correlations — and a value near 1.0 would be
*suspicious*, not reassuring. RNA and protein are genuinely different quantities: transcripts
turn over in hours while protein accumulates over days, and a myelin protein ends up physically
distant from the nucleus that transcribed it. Positive, moderate, and spatially structured is
the right answer here.

The cross-modal agreement is the deliverable of this whole exercise. It is what lets you say the
data is trustworthy enough to build on — and it is only available because all three modalities
were measured on the same section.

---
## Block 5 — Capstone: watch the brain develop (P0 vs P21)

Everything so far used **one** timepoint, P21, where the corpus callosum is fully myelinated. The
atlas this data comes from was built to capture *change over time*. So let's do the experiment
that motivates the whole paper: load the **P0** (day-of-birth) section and put its `Mbp` next to
P21's.

At P0 the brain is barely myelinated — oligodendrocytes have not yet wrapped the callosal axons —
so `Mbp` should be **largely absent**. Its appearance by P21 is the developmental story.

First, we tidy Block 1's loading steps into a single reusable function. This is good practice:
if you find yourself about to copy-paste a block of code to run it on new inputs, wrap it in a
function instead. (P0 has only RNA + coordinates in GEO — no protein or ATAC for this sample —
so this comparison is RNA-only.)

In [ ]:
def load_rna_sample(rna_url, pos_url, tag):
    """Fetch + align one sample's RNA matrix and pixel coordinates.

    Returns (rna_norm_df, xy_array) for that sample. Mirrors Block 1, wrapped for reuse.
    """
    rp = fetch(rna_url, f"data/{tag}_RNA.csv.gz")
    pp = fetch(pos_url, f"data/{tag}_positions.csv.gz")

    r = pd.read_csv(rp, index_col=0).astype(np.int32)
    pos = pd.read_csv(pp, header=None,
                      names=["barcode", "in_tissue", "array_row", "array_col",
                             "pixel_row", "pixel_col"])
    pos = pos[pos.in_tissue == 1].set_index("barcode")

    bc = [b for b in r.columns if b in pos.index]
    r  = r[bc]
    xy_s = pos.loc[bc, ["array_col", "array_row"]].to_numpy(float)
    r_norm = cp10k(r)
    print(f"{tag}: {r.shape[0]:,} genes x {len(bc):,} on-tissue pixels")
    return r_norm, xy_s


def plot_gene_on(ax, rna_norm_df, xy_s, gene, title):
    """Standalone spatial plot for an arbitrary sample (does not rely on the global grid)."""
    if gene not in rna_norm_df.index:
        ax.set_title(f"{gene} — absent"); ax.axis("off"); return
    v = rna_norm_df.loc[gene].to_numpy(float)
    lo, hi = np.quantile(v, 0.02), np.quantile(v, 0.98)
    if hi <= lo:
        hi = lo + 1e-9
    ax.scatter(xy_s[:, 0], -xy_s[:, 1], c=v, s=5.5, marker="s",
               cmap="viridis", vmin=lo, vmax=hi)
    ax.set_title(title); ax.set_aspect("equal"); ax.axis("off")

In [ ]:
P0_RNA = ("https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM9247nnn/GSM9247574/suppl/"
          "GSM9247574_00_P0S1_RNA_matrix.csv.gz")
P0_POS = ("https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM9247nnn/GSM9247574/suppl/"
          "GSM9247574_00_P0S1_tissue_positions_list.csv.gz")

p0_rna, p0_xy = load_rna_sample(P0_RNA, P0_POS, "P0S1")

fig, axes = plt.subplots(1, 2, figsize=(8.4, 4.2))
plot_gene_on(axes[0], p0_rna,    p0_xy, "Mbp", "P0 (newborn) — Mbp")
plot_gene_on(axes[1], rna_norm,  xy,    "Mbp", "P21 (3 weeks) — Mbp")
fig.suptitle("The corpus callosum myelinates between birth and P21", y=1.02)
plt.tight_layout(); plt.show()

print(f"Mbp-positive pixels — P0: {int((p0_rna.loc['Mbp'] > 0).sum()):,}   "
      f"P21: {int((rna_norm.loc['Mbp'] > 0).sum()):,}")

There it is — the single most important result in this notebook, because it is a *difference*,
not a snapshot. At P0 the corpus callosum is dim; by P21 it is the brightest structure in the
section. You have just reproduced, from raw public data, the developmental transition the atlas
was built to measure.

#### 🔧 Your turn — pick another developmental gene

Change `gene` below and compare P0 vs P21. Myelin genes (`Plp1`, `Mog`) tell the same story as
`Mbp`. For contrast, try a pan-neuronal gene like `Snap25`: neurons are already present at birth,
so it should look far more similar between the two ages. *A gene that changes vs a gene that does
not* — that contrast is the whole point of a developmental atlas.

In [ ]:
gene = "Plp1"            # <-- try "Mog", or "Snap25" (present at both ages)
fig, axes = plt.subplots(1, 2, figsize=(8.4, 4.2))
plot_gene_on(axes[0], p0_rna,   p0_xy, gene, f"P0 — {gene}")
plot_gene_on(axes[1], rna_norm, xy,    gene, f"P21 — {gene}")
plt.tight_layout(); plt.show()

---
## Wrap-up

You ran one complete loop of a spatial tri-omics analysis, twice over:

1. **Loaded** three modalities from three different public archives and aligned them on shared
   pixel barcodes.
2. **Checked quality** — depth per pixel, and its spatial distribution.
3. **Confirmed orientation** against a structure whose anatomy you already knew, and overlaid the
   molecular signal on the tissue image itself.
4. **Clustered** the pixels from expression alone, then read the standard chart types — bar,
   heatmap, expression dot plot, and a **GO-enrichment** dot plot — off your own data, recovering
   the oligodendrocyte identity of the white-matter cluster end to end.
5. **Plotted RNA, ATAC and protein on the same pixels**, and handled the sparsity of ATAC by
   aggregating a gene programme.
6. **Validated against known anatomy**, and quantified cross-modal agreement.
7. **Compared P0 and P21** and saw the corpus callosum myelinate — a real developmental result.

**What tri-omics bought us that a single modality could not:** we saw chromatin at the myelin
locus, the transcript, and the protein in one section — accessibility and transcription marking
where myelin genes are *being made*, protein marking where myelin has *accumulated*. Separating
"licensed to express", "expressing now" and "has expressed" is not possible from one modality,
and it is not possible from separate sections either, because you could never be sure you were
comparing the same piece of tissue.

### Going further

- Re-run Block 5 for a different gene, or add the P0 protein/ATAC arms if a future release
  includes them.
- Change `smooth=` anywhere and watch resolution trade against signal.
- Build a programme for a cell type we did not cover (endothelial: `Cldn5`, `Pecam1`, `Flt1`).

### Links

- Paper: [10.1038/s41586-025-09663-y](https://doi.org/10.1038/s41586-025-09663-y)
- RNA: [GEO GSE308526](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE308526) · ATAC: [GEO GSE308599](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE308599)
- Protein: [NeMO Archive](https://assets.nemoarchive.org/col-0cggtum) · Imaging + code: [Zenodo](https://doi.org/10.5281/zenodo.17121652)
- Interactive browser: [spatial-omics.yale.edu](https://spatial-omics.yale.edu/)